In [5]:
import sys
print(sys.executable)

/root/miniconda3/envs/dockingenv/bin/python


In [6]:
import os
os.chdir("/root/Ligand_Prep_trial")

In [7]:
import subprocess
import pandas as pd
import os

# =========================
# CONFIG
# =========================
GNINA = "~/gnina"  # remove the !, subprocess handles execution
MODE = 1  # use pose 1 consistently

# Folders
protein_dir = "proteinprep"
ligand_dir = "ligandprep"
ideal_ligand_dir = "minimized_active_ligand"  # NEW folder for ideal ligands
output_dir = "docked_results_active"

pairs = [
    {"pdb": "1ERE_A_fixed.pdb", "ligand": "EST_redock_1ERE_A.sdf", "ideal": "minimized_ideal_ligand/EST_min.sdf", "lig_id": "EST"},
    {"pdb": "1G50_A_fixed.pdb", "ligand": "EST_redock_1G50_A.sdf", "ideal": "minimized_ideal_ligand/EST_min.sdf", "lig_id": "EST"},
    {"pdb": "1GWR_A_fixed.pdb", "ligand": "EST_redock_1GWR_A.sdf", "ideal": "minimized_ideal_ligand/EST_min.sdf", "lig_id": "EST"},
    {"pdb": "3ERD_A_fixed.pdb", "ligand": "DES_redock_3ERD_A.sdf", "ideal": "minimized_ideal_ligand/DES_min.sdf", "lig_id": "DES"},
    {"pdb": "3UU7_A_fixed.pdb", "ligand": "2OH_redock_3UU7_A.sdf", "ideal": "minimized_ideal_ligand/2OH_min.sdf", "lig_id": "2OH"},
    {"pdb": "3UUD_A_fixed.pdb", "ligand": "EST_redock_3UUD_A.sdf", "ideal": "minimized_ideal_ligand/EST_min.sdf", "lig_id": "EST"},
    {"pdb": "4MG8_A_fixed.pdb", "ligand": "27J_redock_4MG8_A.sdf", "ideal": "minimized_ideal_ligand/27J_min.sdf", "lig_id": "27J"},
    {"pdb": "4MG9_A_fixed.pdb", "ligand": "27K_redock_4MG9_A.sdf", "ideal": "minimized_ideal_ligand/27K_min.sdf", "lig_id": "27K"},
    {"pdb": "4MGA_A_fixed.pdb", "ligand": "27L_redock_4MGA_A.sdf", "ideal": "minimized_ideal_ligand/27L_min.sdf", "lig_id": "27L"},
    {"pdb": "4MGC_A_fixed.pdb", "ligand": "27M_redock_4MGC_A.sdf", "ideal": "minimized_ideal_ligand/27M_min.sdf", "lig_id": "27M"},
     {"pdb": "4TUZ_A_fixed.pdb", "ligand": "36J_redock_4TUZ_A.sdf", "ideal": "minimized_ideal_ligand/36J_min.sdf", "lig_id": "36J"},
     {"pdb": "4ZN7_A_fixed.pdb", "ligand": "DES_redock_4ZN7_A.sdf", "ideal": "minimized_ideal_ligand/DES_min.sdf", "lig_id": "DES"},
     {"pdb": "6CBZ_A_fixed.pdb", "ligand": "EST_redock_6CBZ_A.sdf", "ideal": "minimized_ideal_ligand/EST_min.sdf", "lig_id": "EST"},
]

# =========================
# HELPERS
# =========================
def run_cmd(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("COMMAND FAILED:\n", cmd)
        print(result.stderr)
        raise RuntimeError("Execution stopped")
    return result.stdout


def parse_cnn_from_table(output, mode=1):
    for line in output.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(str(mode)):
            parts = [p for p in line.split() if p.replace('.', '', 1).replace('-', '', 1).isdigit()]
            if len(parts) < 3:
                continue
            vina_affinity = float(parts[1])
            cnn_pose = float(parts[3])
            cnn_affinity = float(parts[4])
            cnn_vs = cnn_pose * cnn_affinity
            return vina_affinity, cnn_pose, cnn_affinity, cnn_vs
    return None, None, None, None


def get_rmsd(ref, sdf, mode=1):
    out = run_cmd(f"obrms -f {ref} {sdf}")
    rmsds = [float(l.split()[-1]) for l in out.splitlines() if l.strip().startswith("RMSD")]
    return rmsds[mode - 1] if len(rmsds) >= mode else None

# =========================
# DOCKING LOOP (ALL IDEALS)
# =========================
results = []

# Get all ideal ligands in folder
all_ideal_sdfs = [f for f in os.listdir(ideal_ligand_dir) if f.endswith(".sdf")]

for p in pairs:
    lig = p["lig_id"]
    print(f"\n🚀 Docking into {p['pdb']}")

    receptor_file = os.path.join(protein_dir, p['pdb'])
    native_ligand_file = os.path.join(ligand_dir, p['ligand'])

    for ideal_name in all_ideal_sdfs:

        ideal_file = os.path.join(ideal_ligand_dir, ideal_name)
        ideal_id = os.path.splitext(ideal_name)[0]

        print(f"   → Ideal ligand: {ideal_name}")

        ideal_out = os.path.join(output_dir, f"docked_{p['pdb']}_{ideal_id}.sdf")

        out = run_cmd(
            f"{GNINA} -r {receptor_file} "
            f"-l {ideal_file} "
            f"--autobox_ligand {native_ligand_file} "
            f"--seed 0 --exhaustiveness 16 "
            f"-o {ideal_out}"
        )

        vina, pose, aff, vs = parse_cnn_from_table(out, MODE)

        # =========================
        # RMSD ONLY FOR EXPLICIT PAIR
        # =========================
        if ideal_name == p["ideal"]:
            rmsd = get_rmsd(native_ligand_file, ideal_out, MODE)
        else:
            rmsd = None

        results.append({
            "ideal_ligand": ideal_id,
            "protein": p["pdb"],
            "ligand_native": lig,
            "dock_type": "ideal_minimized",
            "vina_affinity": vina,
            "CNNpose": pose,
            "CNNaffinity": aff,
            "CNN_VS": vs,
            "RMSD": rmsd
        })

# =========================
# RESULTS
# =========================
df_min = pd.DataFrame(results)

# Sort by ideal_ligand (alphabetical)
df_min = df_min.sort_values(by="ideal_ligand").reset_index(drop=True)

df_min


🚀 Docking into 1ERE_A_fixed.pdb
   → Ideal ligand: Mestranol_min.sdf
   → Ideal ligand: Diethylstilbestrol_min.sdf
   → Ideal ligand: meso-hexestrol_min.sdf
   → Ideal ligand: Dienestrol_min.sdf
   → Ideal ligand: 17alpha-Estradiol_min.sdf
   → Ideal ligand: EE2_min.sdf
   → Ideal ligand: Estrone_min.sdf
   → Ideal ligand: Estriol_min.sdf
   → Ideal ligand: Moxestrol_min.sdf
   → Ideal ligand: EST_min.sdf
   → Ideal ligand: Equilin_min.sdf

🚀 Docking into 1G50_A_fixed.pdb
   → Ideal ligand: Mestranol_min.sdf
   → Ideal ligand: Diethylstilbestrol_min.sdf
   → Ideal ligand: meso-hexestrol_min.sdf
   → Ideal ligand: Dienestrol_min.sdf
   → Ideal ligand: 17alpha-Estradiol_min.sdf
   → Ideal ligand: EE2_min.sdf
   → Ideal ligand: Estrone_min.sdf
   → Ideal ligand: Estriol_min.sdf
   → Ideal ligand: Moxestrol_min.sdf
   → Ideal ligand: EST_min.sdf
   → Ideal ligand: Equilin_min.sdf

🚀 Docking into 1GWR_A_fixed.pdb
   → Ideal ligand: Mestranol_min.sdf
   → Ideal ligand: Diethylstilbestrol_mi

,ideal_ligand,protein,ligand_native,dock_type,vina_affinity,CNNpose,CNNaffinity,CNN_VS,RMSD
0,17alpha-Estradiol_min,4MG8_A_fixed.pdb,27J,ideal_minimized,-11.12,0.9672,7.769,7.514177,None
1,17alpha-Estradiol_min,4TUZ_A_fixed.pdb,36J,ideal_minimized,-10.83,0.9556,8.202,7.837831,None
2,17alpha-Estradiol_min,3UU7_A_fixed.pdb,2OH,ideal_minimized,-8.76,0.8648,7.937,6.863918,None
3,17alpha-Estradiol_min,4ZN7_A_fixed.pdb,DES,ideal_minimized,-11.22,0.9749,8.240,8.033176,None
4,17alpha-Estradiol_min,1G50_A_fixed.pdb,EST,ideal_minimized,-11.83,0.9710,8.188,7.950548,None
...,...,...,...,...,...,...,...,...,...
138,meso-hexestrol_min,4MGC_A_fixed.pdb,27M,ideal_minimized,-8.85,0.9624,8.295,7.983108,None
139,meso-hexestrol_min,3ERD_A_fixed.pdb,DES,ideal_minimized,-9.33,0.9879,8.314,8.213401,None
140,meso-hexestrol_min,4TUZ_A_fixed.pdb,36J,ideal_minimized,-8.28,0.9108,7.908,7.202606,None
141,meso-hexestrol_min,4MG9_A_fixed.pdb,27K,ideal_minimized,-8.55,0.9532,8.000,7.625600,None


In [8]:
df_min.to_excel("results/docking_results_ER_ACTIVE_Minimized Ligand.xlsx", index=False)

In [23]:
input_file = "generated_decoys_activeER.sdf"
output_file = "test_all.sdf"

start = 0
end = 482

mol_count = 0
write = False

with open(input_file, "r") as fin, open(output_file, "w") as fout:
    for line in fin:
        # check start of writing
        if mol_count == start:
            write = True

        # write lines only if within range
        if write:
            fout.write(line)

        # detect end of molecule
        if line.strip() == "$$$$":
            mol_count += 1

            # stop after end index
            if mol_count > end:
                break

In [35]:
import os
import subprocess
import pandas as pd

GNINA = "~/gnina"

protein_dir = "proteinprep"
ligand_dir = "ligandprep"
output_dir = "docked_results_decoy"
tmp_dir = "tmp_ligands"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(tmp_dir, exist_ok=True)

# ---------------------------
# Protein set
# ---------------------------
proteins = [
    {"pdb": "1ERE_A_fixed.pdb", "ref_lig": "EST_redock_1ERE_A.sdf"},
    {"pdb": "1G50_A_fixed.pdb", "ref_lig": "EST_redock_1G50_A.sdf"},
    {"pdb": "1GWR_A_fixed.pdb", "ref_lig": "EST_redock_1GWR_A.sdf"},
    {"pdb": "3ERD_A_fixed.pdb", "ref_lig": "DES_redock_3ERD_A.sdf"},
    {"pdb": "3UU7_A_fixed.pdb", "ref_lig": "2OH_redock_3UU7_A.sdf"},
    {"pdb": "3UUD_A_fixed.pdb", "ref_lig": "EST_redock_3UUD_A.sdf"},
    {"pdb": "4MG8_A_fixed.pdb", "ref_lig": "27J_redock_4MG8_A.sdf"},
    {"pdb": "4MG9_A_fixed.pdb", "ref_lig": "27K_redock_4MG9_A.sdf"},
    {"pdb": "4MGA_A_fixed.pdb", "ref_lig": "27L_redock_4MGA_A.sdf"},
    {"pdb": "4MGC_A_fixed.pdb", "ref_lig": "27M_redock_4MGC_A.sdf"},
    {"pdb": "4TUZ_A_fixed.pdb", "ref_lig": "36J_redock_4TUZ_A.sdf"},
    {"pdb": "4ZN7_A_fixed.pdb", "ref_lig": "DES_redock_4ZN7_A.sdf"},
    {"pdb": "6CBZ_A_fixed.pdb", "ref_lig": "EST_redock_6CBZ_A.sdf"},
]

multi_ligand_file = "generated_decoys_activeER_filtered_FINAL.sdf"


# ---------------------------
# Helpers
# ---------------------------
def run_cmd(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("COMMAND FAILED:\n", cmd)
        print(result.stderr)
        raise RuntimeError("Execution stopped")
    return result.stdout


def extract_ligands(sdf_file):
    """Split SDF into list of full ligand blocks."""
    ligands = []
    current = []

    with open(sdf_file) as f:
        for line in f:
            current.append(line)
            if line.strip() == "$$$$":
                ligands.append("".join(current))
                current = []
    return ligands


def parse_gnina_sdf(filepath, pdb_id, ligand_id):
    """Parse gnina SDF output for all poses."""
    records = []

    with open(filepath, "r") as f:
        content = f.read()

    mols = content.split("$$$$")

    for mol in mols:
        if "<CNNscore>" not in mol:
            continue

        def grab(field):
            if f"> <{field}>" in mol:
                return float(
                    mol.split(f"> <{field}>")[1].split("\n")[1].strip()
                )
            return None

        records.append({
            "pdb": pdb_id,
            "ligand_id": ligand_id,
            "CNNscore": grab("CNNscore"),
            "CNNaffinity": grab("CNNaffinity"),
            "vina_affinity": grab("minimizedAffinity"),
        })

    return records


# ---------------------------
# Load ligands once
# ---------------------------
ligands = extract_ligands(multi_ligand_file)


# ---------------------------
# Docking loop (ONE ligand at a time)
# ---------------------------
results = []

for p in proteins:
    print(f"\n🚀 Docking into {p['pdb']}")

    receptor = os.path.join(protein_dir, p["pdb"])
    ref_lig = os.path.join(ligand_dir, p["ref_lig"])

    for i, lig in enumerate(ligands):
        lig_file = os.path.join(tmp_dir, f"lig_{i}.sdf")
        output_sdf = os.path.join(output_dir, f"{p['pdb']}_lig_{i}.sdf")

        # write single ligand
        with open(lig_file, "w") as f:
            f.write(lig)

        # run gnina
        cmd = (
            f"{GNINA} "
            f"-r {receptor} "
            f"-l {lig_file} "
            f"--autobox_ligand {ref_lig} "
            f"--num_modes 9 "
            f"--seed 0 "
            f"--exhaustiveness 16 "
            f"-o {output_sdf}"
        )

        run_cmd(cmd)

        # parse results for this ligand
        results.extend(
            parse_gnina_sdf(output_sdf, p["pdb"], i)
        )


# ---------------------------
# Build dataframe
# ---------------------------
df = pd.DataFrame(results)


# ---------------------------
# Select BEST pose per ligand
# ---------------------------
df_top = (
    df.loc[df.groupby(["pdb", "ligand_id"])["CNNscore"].idxmax()]
    .reset_index(drop=True)
)


# ---------------------------
# Save final results
# ---------------------------
df_top.to_csv("gnina_top_decoys_single_ligand_ALLER.csv", index=False)

print("✅ Done:", len(df_top), "ligands retained")


🚀 Docking into 1ERE_A_fixed.pdb

🚀 Docking into 1G50_A_fixed.pdb

🚀 Docking into 1GWR_A_fixed.pdb

🚀 Docking into 3ERD_A_fixed.pdb

🚀 Docking into 3UU7_A_fixed.pdb

🚀 Docking into 3UUD_A_fixed.pdb

🚀 Docking into 4MG8_A_fixed.pdb

🚀 Docking into 4MG9_A_fixed.pdb

🚀 Docking into 4MGA_A_fixed.pdb

🚀 Docking into 4MGC_A_fixed.pdb

🚀 Docking into 4TUZ_A_fixed.pdb

🚀 Docking into 4ZN7_A_fixed.pdb

🚀 Docking into 6CBZ_A_fixed.pdb
✅ Done: 5642 ligands retained
